# HHT variant curation (ENG, ACVRL1, SMAD4, GDF2)

Runs in the All of Us Researcher Workbench (Python kernel on a **Hail / Dataproc** environment).

| Step | What happens | Output (workspace bucket) |
|---|---|---|
| 1 | Pull srWGS exome-callset genotypes for the 4 gene regions | `gene_region.mt` (checkpoint) |
| 2 | Sample + genotype QC, per-variant counts | — |
| 3 | Carrier table for rare variants (person × variant) | `carriers.tsv` |
| 4 | Annotate unique variants: VAT (consequence, HGVS, gnomAD, AoU AF, REVEL, SpliceAI) + current ClinVar (with review stars) | `variants_annotated.tsv` |
| 5 | Automatic filters and tiers (T1 ClinVar P/LP, T2 predicted LoF, T3 rare damaging missense/splice/inframe) | `variants_tiered.tsv` |
| 6 | **Manual curation** (ACMG/AMP + ClinGen HHT VCEP) in the curation table, in the Workbench or offline from a downloadable export | `curation_table.tsv`, `curation_export_<date>.tsv` |
| 7 | Join final classes back to carriers → genotype-positive cohort | `hht_genotype_positive.tsv` |

**Data policy.** Do not download `carriers.tsv` or any table with participant IDs, genotype counts or All of Us allele frequencies. The only downloadable output is the step 6a export: the candidate variants with their annotations (VAT, ClinVar, gnomAD), with every All of Us count and frequency column removed. Any count 1–20 must be suppressed before it leaves the Workbench (`mask()` below). Franklin/ClinVar/ARUP look-ups are done one variant at a time, by coordinates only; never upload files.

Re-running is safe: expensive steps are checkpointed, and step 6 keeps whatever you already entered in the curation table.

## 0. Setup

In [ ]:
import os, io, re, gzip, json, subprocess, urllib.request
from datetime import date
import numpy as np
import pandas as pd
import hail as hl

PROJECT = os.environ.get("GOOGLE_PROJECT") or os.environ.get("GOOGLE_CLOUD_PROJECT") or "wb-silky-artichoke-2408"
BUCKET  = os.environ.get("WORKSPACE_BUCKET") or "gs://hht-epi-bucket-wb-lucent-greens-2488"
CDR     = os.environ.get("WORKSPACE_CDR") or "wb-silky-artichoke-2408.C2025Q4R6"
OUT     = f"{BUCKET}/hht_variant_curation"      # all outputs go here
LOCAL   = "hht_variant_curation"                  # local scratch on the notebook VM
os.makedirs(LOCAL, exist_ok=True)

try:
    hl.init(default_reference="GRCh38", idempotent=True, gcs_requester_pays_configuration=PROJECT)
except TypeError:  # older Hail without the requester-pays argument
    hl.init(default_reference="GRCh38", idempotent=True)

print("project:", PROJECT, "\nbucket :", BUCKET, "\nCDR    :", CDR, "\nhail   :", hl.version())

In [ ]:
def gs(*args):
    # gsutil with requester-pays billing; returns stdout
    r = subprocess.run(["gsutil", "-u", PROJECT, *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.strip())
    return r.stdout

def read_gs_table(path, **kw):
    return pd.read_csv(io.StringIO(gs("cat", path)), sep="\t", **kw)

def save(df, name):
    # write locally, then copy to the workspace bucket
    local = f"{LOCAL}/{name}"
    df.to_csv(local, sep="\t", index=False)
    gs("cp", local, f"{OUT}/{name}")
    print(f"saved {len(df):,} rows -> {OUT}/{name}")

def load(name):
    local = f"{LOCAL}/{name}"
    gs("cp", f"{OUT}/{name}", local)
    return pd.read_csv(local, sep="\t", dtype={"person_id": str})

def mask(n):
    # All of Us dissemination policy: no counts 1-20 outside the Workbench
    return "<=20" if 1 <= n <= 20 else str(n)

## 1. Configuration

**Check the data paths before the first run.** They change between CDR releases and between the old and new Researcher Workbench. Take them from the CDR v9 Data Dictionary / Controlled CDR Directory, or use the discovery cell below. The genotype path is read from the `WGS_EXOME_SPLIT_HAIL_PATH` environment variable when the workspace sets it.

In [ ]:
# --- data paths (verify!) ---------------------------------------------------
EXOME_SPLIT_MT = os.environ.get("WGS_EXOME_SPLIT_HAIL_PATH", "FILL_ME: .../wgs/short_read/snpindel/exome/splitMT/hail.mt")
AUX            = "FILL_ME: .../wgs/short_read/snpindel/aux"   # folder with vat/, qc/, relatedness/, ancestry/
VAT_PATH       = f"{AUX}/vat/vat_complete.bgz.tsv.gz"          # check exact file name
FLAGGED_PATH   = f"{AUX}/qc/flagged_samples.tsv"                # samples failing joint-callset QC
RELATED_PATH   = f"{AUX}/relatedness/relatedness_flagged_samples.tsv"  # optional: max unrelated set complement
ANCESTRY_PATH  = f"{AUX}/ancestry/ancestry_preds.tsv"           # optional

VAT_SOURCE = "tsv"        # "tsv" (Hail scan of the VAT file) or "bigquery" (if a VAT table exists in the CDR, see discovery)
VAT_BQ_TABLE = None       # e.g. f"{CDR}.<vat_table_name>" when VAT_SOURCE == "bigquery"

# --- genes: GRCh38 coordinates and MANE Select transcripts (Ensembl REST, checked 2026-10) ---
GENES = {
    #         chrom    start        end         MANE Ensembl         MANE RefSeq
    "ENG":    ("chr9",  127811130, 127855232, "ENST00000373203", "NM_001114753.3"),
    "ACVRL1": ("chr12",  51906908,  51923361, "ENST00000388922", "NM_000020.3"),
    "SMAD4":  ("chr18",  51028528,  51085045, "ENST00000342988", "NM_005359.6"),
    "GDF2":   ("chr10",  47322451,  47327592, "ENST00000581492", "NM_016204.4"),
}
PAD = 2_000               # bp around each gene (promoter/UTR/splice margins)
LOF_GENES = {"ENG", "ACVRL1", "SMAD4"}   # GDF2 LoF goes to T3 (weaker gene-disease evidence)

# --- thresholds --------------------------------------------------------------
MIN_GQ, MIN_DP, AB_MIN, AB_MAX = 20, 10, 0.25, 0.75
CARRIER_AF_MAX    = 1e-3   # only rarer variants get person-level carrier rows
AF_MAX            = 1e-4   # popmax screen for "rare" (replace with HHT VCEP BS1/BA1 values)
CLINVAR_MIN_STARS = 1
REVEL_MIN         = 0.644  # ClinGen-calibrated PP3 supporting (Pejaver 2022)
SPLICEAI_MIN      = 0.5

In [ ]:
# Discovery: what genomic env vars / files / BigQuery VAT tables exist in this workspace?
print("\n".join(f"{k} = {v}" for k, v in sorted(os.environ.items()) if k.startswith(("WGS", "CDR", "WORKSPACE", "GOOGLE"))))

if not AUX.startswith("FILL_ME"):
    for sub in ["", "vat", "qc", "relatedness", "ancestry"]:
        try: print(f"\n# {AUX}/{sub}\n" + gs("ls", f"{AUX}/{sub}"))
        except RuntimeError as e: print(f"\n# {AUX}/{sub}: {e}")

from google.cloud import bigquery
bq = bigquery.Client()
proj, ds = CDR.split(".")
q = f'''
SELECT table_name, STRING_AGG(column_name, ', ' ORDER BY ordinal_position) AS columns
FROM `{proj}.{ds}.INFORMATION_SCHEMA.COLUMNS`
WHERE REGEXP_CONTAINS(LOWER(table_name), r'vat|variant')
GROUP BY table_name'''
try:
    print(bq.query(q).to_dataframe().to_string())
except Exception as e:
    print("BigQuery schema lookup failed:", e)

In [ ]:
assert not EXOME_SPLIT_MT.startswith("FILL_ME"), "Set EXOME_SPLIT_MT (see discovery cell / Data Dictionary)"
assert not AUX.startswith("FILL_ME"), "Set AUX (see discovery cell / Data Dictionary)"
INTERVALS = [hl.parse_locus_interval(f"{c}:{s - PAD}-{e + PAD}", reference_genome="GRCh38")
             for c, s, e, *_ in GENES.values()]
MANE = {g: v[3] for g, v in GENES.items()}

## 2. Genotypes for the gene regions, with sample and genotype QC

In [ ]:
mt = hl.read_matrix_table(EXOME_SPLIT_MT)
mt = hl.filter_intervals(mt, INTERVALS)
mt = mt.checkpoint(f"{OUT}/gene_region.mt", _read_if_exists=True)
mt.describe()
print("variants x samples:", mt.count())

In [ ]:
# drop samples that failed joint-callset QC
flagged = read_gs_table(FLAGGED_PATH, dtype=str)
flag_ids = set(flagged.iloc[:, 0])
print(f"flagged samples in file: {len(flag_ids):,}")
mt = mt.filter_cols(~hl.literal(flag_ids).contains(mt.s))
print("samples after removal:", mt.count_cols())

In [ ]:
# genotype QC applies to non-reference calls; field names differ between releases, so build it from what exists
E = mt.entry
def ft_pass():
    if "FT" not in E: return hl.bool(True)
    t = mt.FT.dtype
    if t == hl.tbool: return hl.or_else(mt.FT, True)
    if t == hl.tstr:  return hl.is_missing(mt.FT) | hl.literal({"PASS", "."}).contains(mt.FT)
    return hl.is_missing(mt.FT) | (hl.len(mt.FT) == 0) | mt.FT.contains("PASS")

gq = mt.GQ if "GQ" in E else hl.missing(hl.tint32)
ad = mt.AD if "AD" in E else hl.missing(hl.tarray(hl.tint32))
dp = mt.DP if "DP" in E else hl.sum(ad)
mt = mt.annotate_entries(GQ_=gq, DP_=dp, AB_=hl.or_missing(hl.sum(ad) > 0, ad[1] / hl.sum(ad)))
qc = (ft_pass() & (mt.GQ_ >= MIN_GQ) & (mt.DP_ >= MIN_DP)
      & (~mt.GT.is_het() | ((mt.AB_ >= AB_MIN) & (mt.AB_ <= AB_MAX))))
mt = mt.annotate_entries(qc=hl.or_else(qc, False))

mt = mt.annotate_rows(
    n_called = hl.agg.count_where(hl.is_defined(mt.GT) & (mt.GT.is_hom_ref() | mt.qc)),
    n_het    = hl.agg.count_where(mt.GT.is_het() & mt.qc),
    n_hom    = hl.agg.count_where(mt.GT.is_hom_var() & mt.qc),
    n_fail   = hl.agg.count_where(mt.GT.is_non_ref() & ~mt.qc),
)
mt = mt.annotate_rows(af_aou=hl.or_missing(mt.n_called > 0, (mt.n_het + 2 * mt.n_hom) / (2 * mt.n_called)))
mt = mt.filter_rows((mt.n_het + mt.n_hom + mt.n_fail) > 0)

In [ ]:
# one row per variant (small: a few thousand)
rows = mt.rows()
rows = rows.select(contig=rows.locus.contig, position=rows.locus.position,
                   ref=rows.alleles[0], alt=rows.alleles[1],
                   n_called=rows.n_called, n_het=rows.n_het, n_hom=rows.n_hom,
                   n_fail_qc=rows.n_fail, af_aou=rows.af_aou)
variants = rows.key_by().drop("locus", "alleles").to_pandas()
variants["vkey"] = variants.contig + "-" + variants.position.astype(str) + "-" + variants.ref + "-" + variants.alt
print(f"variants with >=1 carrier: {len(variants):,}")
print(f"  rare (AoU AF <= {CARRIER_AF_MAX}): {(variants.af_aou <= CARRIER_AF_MAX).sum():,}")

## 3. Carrier table (rare variants only; never leaves the Workbench)

In [ ]:
rare = mt.filter_rows(mt.af_aou <= CARRIER_AF_MAX)
rare = rare.filter_entries(rare.GT.is_non_ref())
ent = rare.entries()
ent = ent.key_by().select(
    person_id=ent.s,
    vkey=hl.delimit([ent.locus.contig, hl.str(ent.locus.position), ent.alleles[0], ent.alleles[1]], "-"),
    GT=hl.str(ent.GT), GQ=ent.GQ_, DP=ent.DP_, AB=hl.float(ent.AB_), qc_pass=ent.qc)
carriers = ent.to_pandas()
carriers["person_id"] = carriers.person_id.astype(str)
save(carriers, "carriers.tsv")
print(f"carrier rows: {len(carriers):,}  (QC-pass: {carriers.qc_pass.sum():,})")

## 4. Annotate unique variants

**VAT** fields used (from the VAT schema): consequence, HGVS c./p., exon number, gnomAD and All of Us popmax AF, REVEL, SpliceAI, ClinVar classification at VAT build time. The VAT has one row per variant × transcript; the MANE Select transcript is kept, falling back to the canonical transcript.

The first VAT scan reads the whole file and can take a while. Use a larger cluster for this one run; the result is cached in the bucket.

In [ ]:
VAT_COLS = ["contig", "position", "ref_allele", "alt_allele", "gene_symbol", "transcript", "is_canonical_transcript",
            "consequence", "dna_change_in_transcript", "aa_change", "exon_number", "intron_number", "variant_type",
            "dbsnp_rsid", "gnomad_all_af", "gnomad_max_af", "gnomad_max_subpop", "gvs_all_af", "gvs_max_af",
            "gvs_max_subpop", "revel", "splice_ai_acceptor_gain_score", "splice_ai_acceptor_loss_score",
            "splice_ai_donor_gain_score", "splice_ai_donor_loss_score", "clinvar_classification",
            "clinvar_last_updated", "clinvar_phenotype"]

if VAT_SOURCE == "bigquery":
    vat = bq.query(f'''SELECT * FROM `{VAT_BQ_TABLE}`
                       WHERE gene_symbol IN UNNEST(@genes)''',
                   job_config=bigquery.QueryJobConfig(query_parameters=[
                       bigquery.ArrayQueryParameter("genes", "STRING", list(GENES))])).to_dataframe()
else:
    ht = hl.import_table(VAT_PATH, force_bgz=True, missing="")
    ht = ht.filter(hl.literal(set(GENES)).contains(ht.gene_symbol))
    ht = ht.select(*[c for c in VAT_COLS if c in ht.row])
    ht = ht.checkpoint(f"{OUT}/vat_hht_genes.ht", _read_if_exists=True)
    vat = ht.to_pandas()

vat = vat[[c for c in VAT_COLS if c in vat.columns]].copy()
for c in vat.columns:
    if any(k in c for k in ("_af", "revel", "_score", "position")):
        vat[c] = pd.to_numeric(vat[c], errors="coerce")
vat = vat.astype({c: str for c in ["consequence", "clinvar_classification", "clinvar_phenotype", "dbsnp_rsid"] if c in vat})
vat["vkey"] = vat.contig + "-" + vat.position.astype("Int64").astype(str) + "-" + vat.ref_allele + "-" + vat.alt_allele
vat["enst"] = vat.transcript.astype(str).str.split(".").str[0]
vat["is_mane"] = vat.enst == vat.gene_symbol.map(MANE)
canon = vat.is_canonical_transcript.astype(str).str.lower().isin(["true", "1"])
vat["pick"] = np.where(vat.is_mane, 0, np.where(canon, 1, 2))
vat1 = (vat.sort_values(["vkey", "pick"])
           .drop_duplicates("vkey")
           .rename(columns={"dna_change_in_transcript": "hgvs_c", "aa_change": "hgvs_p",
                            "clinvar_classification": "clinvar_vat"}))
vat1["transcript_used"] = vat1.pick.map({0: "MANE", 1: "canonical", 2: "other"})
sa = [c for c in vat1.columns if c.startswith("splice_ai_") and c.endswith("_score")]
vat1["spliceai_max"] = vat1[sa].max(axis=1)
print(f"VAT rows for the 4 genes: {len(vat):,}; unique variants: {len(vat1):,}")
print(vat1.transcript_used.value_counts())

**Current ClinVar** (the VAT's ClinVar copy is frozen at build time and has no review status). Downloads the GRCh38 VCF from NCBI. If the notebook has no internet access, upload `clinvar.vcf.gz` to `OUT` and point `CLINVAR_LOCAL` at a local copy.

In [ ]:
CLINVAR_URL   = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz"
CLINVAR_LOCAL = f"{LOCAL}/clinvar.vcf.gz"
if not os.path.exists(CLINVAR_LOCAL):
    urllib.request.urlretrieve(CLINVAR_URL, CLINVAR_LOCAL)

STARS = {"practice_guideline": 4, "reviewed_by_expert_panel": 3,
         "criteria_provided,_multiple_submitters,_no_conflicts": 2,
         "criteria_provided,_conflicting_classifications": 1,
         "criteria_provided,_conflicting_interpretations": 1,
         "criteria_provided,_single_submitter": 1}
regions = {c.replace("chr", ""): [] for c, *_ in GENES.values()}
for c, s, e, *_ in GENES.values():
    regions[c.replace("chr", "")].append((s - PAD, e + PAD))

recs, clinvar_date = [], None
with gzip.open(CLINVAR_LOCAL, "rt") as f:
    for line in f:
        if line.startswith("##fileDate="): clinvar_date = line.strip().split("=")[1]
        if line[0] == "#": continue
        chrom, pos, vcv, ref, alt, _, _, info = line.rstrip("\n").split("\t")[:8]
        if chrom not in regions: continue
        pos = int(pos)
        if not any(s <= pos <= e for s, e in regions[chrom]): continue
        d = dict(kv.split("=", 1) for kv in info.split(";") if "=" in kv)
        for a in alt.split(","):
            recs.append({"vkey": f"chr{chrom}-{pos}-{ref}-{a}", "clinvar_id": vcv,
                         "clinvar_sig": d.get("CLNSIG", ""), "clinvar_sig_conf": d.get("CLNSIGCONF", ""),
                         "clinvar_revstat": d.get("CLNREVSTAT", ""), "clinvar_dn": d.get("CLNDN", "")})
clinvar = pd.DataFrame(recs).drop_duplicates("vkey")
clinvar["clinvar_stars"] = clinvar.clinvar_revstat.map(STARS).fillna(0).astype(int)
gs("cp", CLINVAR_LOCAL, f"{OUT}/clinvar_{clinvar_date}.vcf.gz")   # keep the exact release used
print(f"ClinVar release {clinvar_date}: {len(clinvar):,} records in the gene regions")

In [ ]:
ann = variants.merge(vat1.drop(columns=["contig", "position"]), on="vkey", how="left") \
              .merge(clinvar, on="vkey", how="left")
print("matched to VAT    :", ann.gene_symbol.notna().sum(), "/", len(ann))
print("matched to ClinVar:", ann.clinvar_id.notna().sum())
ann["gene"] = ann.gene_symbol.fillna(
    ann.apply(lambda r: next((g for g, (c, s, e, *_) in GENES.items()
                              if r.contig == c and s - PAD <= r.position <= e + PAD), None), axis=1))
save(ann, "variants_annotated.tsv")

## 5. Automatic filters and tiers

| Tier | Rule |
|---|---|
| **T1** | Current ClinVar P/LP, ≥ `CLINVAR_MIN_STARS` stars, no conflicts |
| **T2** | Predicted LoF (stop-gained, frameshift, canonical splice, start-lost) in ENG/ACVRL1/SMAD4, rare |
| **T3** | Rare missense with REVEL ≥ 0.644, rare in-frame indel/stop-lost, SpliceAI ≥ 0.5, or GDF2 LoF |
| check | ClinVar P/LP that fails a T1 condition (0 stars, conflict, or too common): review why |

*Rare* means gnomAD popmax and All of Us AF both ≤ `AF_MAX`. Variants that ClinVar calls B/LB with ≥2 stars are excluded from T2/T3. The VAT has no LOFTEE, so LoF calls in the last exon are flagged for manual NMD review instead of being dropped.

In [ ]:
v = ann.copy()
csq = v.consequence.fillna("").str.lower()
sig = v.clinvar_sig.fillna("").str.lower()
LOF = r"stop_gained|frameshift|splice_acceptor_variant|splice_donor_variant|start_lost|transcript_ablation"
MIS = r"missense|inframe_insertion|inframe_deletion|protein_altering|stop_lost"

v["is_lof"]       = csq.str.contains(LOF)
v["is_missense"]  = csq.str.contains(MIS) & ~v.is_lof
v["popmax_af"]    = v[["gnomad_max_af", "gvs_max_af", "af_aou"]].max(axis=1).fillna(0)
v["rare"]         = v.popmax_af <= AF_MAX
v["clinvar_plp"]  = sig.str.contains("pathogenic") & ~sig.str.contains("conflicting|uncertain|benign")
v["clinvar_blb"]  = sig.str.contains("benign") & ~sig.str.contains("pathogenic|conflicting")
v["benign_strong"] = v.clinvar_blb & (v.clinvar_stars.fillna(0) >= 2)
ex = v.exon_number.fillna("").str.extract(r"(\d+)/(\d+)").astype(float)
v["last_exon"]    = v.is_lof & (ex[0] == ex[1])

t1 = v.clinvar_plp & (v.clinvar_stars >= CLINVAR_MIN_STARS)
t2 = v.is_lof & v.gene.isin(LOF_GENES) & v.rare & ~v.benign_strong
t3 = v.rare & ~v.benign_strong & (
        (v.is_missense & ((v.revel >= REVEL_MIN) | v.revel.isna()))   # REVEL missing = in-frame indels etc.
      | (v.spliceai_max >= SPLICEAI_MIN)
      | (v.is_lof & (v.gene == "GDF2")))
check = v.clinvar_plp & ~t1

v["tier"] = np.select([t1, t2, t3, check], ["T1", "T2", "T3", "check"], default="")
flags = []
for _, r in v.iterrows():
    f = []
    if r.clinvar_plp and not r.rare: f.append("common_for_HHT")
    if r.last_exon: f.append("last_exon_NMD?")
    if r.transcript_used != "MANE": f.append(f"not_MANE({r.transcript_used})")
    if r.n_fail_qc > 0: f.append("some_calls_fail_QC")
    if r.tier == "T1" and not r.rare: f.append("T1_but_frequent")
    flags.append(";".join(f))
v["flags"] = flags
save(v, "variants_tiered.tsv")

summary = v[v.tier != ""].groupby(["gene", "tier"]).size().unstack(fill_value=0)
print("variants to curate (internal counts; mask before sharing):")
print(summary)

## 6. Curation table

One row per candidate variant. Fill in the blank columns either

- **inside the Workbench**: open `hht_variant_curation/curation_table.tsv` in the JupyterLab editor, or edit with pandas, then re-run the save cell; or
- **offline**: download the export from step 6a, fill it in, upload it and run step 6b.

Re-running this step adds new candidates and keeps everything you already entered.

- `final_class`: P, LP, VUS, LB, B
- `acmg_criteria`: e.g. `PVS1, PM2_Supporting, PP4`; apply the ClinGen HHT VCEP specifications for ENG/ACVRL1 (note the spec version)
- `franklin`, `arup_hht_db`, `clinvar_notes`: what each source showed, and why you agree or not

In [ ]:
SHOW = ["vkey", "gene", "tier", "flags", "hgvs_c", "hgvs_p", "consequence", "exon_number",
        "popmax_af", "gnomad_max_af", "gnomad_max_subpop", "gvs_max_af", "af_aou",
        "revel", "spliceai_max", "clinvar_id", "clinvar_sig", "clinvar_stars", "clinvar_revstat",
        "clinvar_vat", "n_het", "n_hom", "n_fail_qc"]
MANUAL = ["final_class", "acmg_criteria", "vcep_spec_version", "franklin", "arup_hht_db",
          "clinvar_notes", "rationale", "reviewer", "review_date"]

cand = v.loc[v.tier != "", SHOW].copy()
cand["tier"] = pd.Categorical(cand.tier, ["T1", "T2", "T3", "check"])
cand = cand.sort_values(["gene", "tier", "vkey"])

try:
    prev = load("curation_table.tsv")
    cand = cand.merge(prev[["vkey"] + [c for c in MANUAL if c in prev]], on="vkey", how="left")
    dropped = set(prev.vkey) - set(cand.vkey)
    if dropped: print(f"{len(dropped)} previously curated variants are no longer candidates:", sorted(dropped)[:10])
except Exception:
    pass
for c in MANUAL:
    if c not in cand: cand[c] = ""
save(cand, "curation_table.tsv")
print(cand.groupby("tier", observed=True).size())

### 6a. Export the candidate variants for offline curation

Writes `curation_export_<date>.tsv` (and `.xlsx` when `openpyxl` is installed) with one row per candidate variant: coordinates, HGVS, consequence, REVEL, SpliceAI, gnomAD, ClinVar and the blank curation columns. Also writes `vat_candidates_all_transcripts.tsv` with every VAT transcript row for those variants.

**Removed before export:** person IDs, genotype counts (`n_het`, `n_hom`, `n_called`, `n_fail_qc`) and every All of Us allele frequency (`af_aou`, `gvs_*`, and `popmax_af`, which includes the All of Us AF). The cell stops if any of them would be written. gnomAD frequencies are external public data and are kept.

To download: in the JupyterLab file browser, open `hht_variant_curation/`, right-click the export file and choose **Download**.

In [ ]:
AOU_DERIVED = {"person_id", "n_called", "n_het", "n_hom", "n_fail_qc", "af_aou", "popmax_af"}
def aou_derived(col):
    return col in AOU_DERIVED or col.startswith("gvs_")

EXTRA = ["transcript_used", "intron_number", "dbsnp_rsid", "gnomad_all_af", "clinvar_sig_conf",
         "clinvar_dn", "clinvar_vat", "clinvar_phenotype", "clinvar_last_updated"] \
        + [c for c in v.columns if c.startswith("splice_ai_")]
exp = cand.merge(v[["vkey"] + [c for c in EXTRA if c in v and c not in cand]], on="vkey", how="left")
exp = exp[[c for c in exp.columns if not aou_derived(c)]]
leaked = [c for c in exp.columns if aou_derived(c)]
assert not leaked, f"All of Us-derived columns in export: {leaked}"

stamp = date.today().isoformat()
EXPORT_NAME = f"curation_export_{stamp}.tsv"
save(exp, EXPORT_NAME)
try:
    exp.to_excel(f"{LOCAL}/curation_export_{stamp}.xlsx", index=False)
    print(f"also wrote {LOCAL}/curation_export_{stamp}.xlsx")
except ImportError:
    print("openpyxl not installed; TSV only")

# every VAT transcript row for the candidate variants (needs `vat` from step 4 in this session)
vat_exp = vat[vat.vkey.isin(exp.vkey)]
vat_exp = vat_exp[[c for c in vat_exp.columns if not aou_derived(c)]]
save(vat_exp, "vat_candidates_all_transcripts.tsv")
print(f"columns exported: {', '.join(exp.columns)}")

### 6b. Import the completed curation

Upload the filled-in file into `hht_variant_curation/` (JupyterLab file browser, **Upload** button), set `CURATED_UPLOAD` to its name, and run the cell. Only `vkey` and the manual columns are read; the other columns are ignored, so it does not matter if Excel reformatted them. Non-empty uploaded values overwrite what is in `curation_table.tsv`; empty cells leave existing entries alone.

In [ ]:
CURATED_UPLOAD = f"{LOCAL}/curation_export_completed.xlsx"   # name of the uploaded file (.tsv or .xlsx)

up = (pd.read_excel(CURATED_UPLOAD, dtype=str) if CURATED_UPLOAD.endswith((".xlsx", ".xls"))
      else pd.read_csv(CURATED_UPLOAD, sep="\t", dtype=str)).fillna("")
up = up[["vkey"] + [c for c in MANUAL if c in up]]
up["vkey"] = up.vkey.str.strip()
assert up.vkey.is_unique, "duplicate vkey rows in the uploaded file"

cur_tab = load("curation_table.tsv")
unknown = set(up.vkey) - set(cur_tab.vkey)
assert not unknown, f"{len(unknown)} vkeys are not in curation_table.tsv, e.g. {sorted(unknown)[:5]}"
up["final_class"] = up.final_class.str.upper().str.strip()
bad = set(up.final_class) - {"", "P", "LP", "VUS", "LB", "B"}
assert not bad, f"unexpected final_class values: {bad}"

cur_tab = cur_tab.set_index("vkey")
cur_tab[MANUAL] = cur_tab[MANUAL].fillna("").astype(str)
upd = up.set_index("vkey")
n_set = 0
for c in [c for c in MANUAL if c in upd]:
    vals = upd[c].str.strip()
    vals = vals[vals != ""]
    cur_tab.loc[vals.index, c] = vals
    n_set += len(vals)
save(cur_tab.reset_index(), "curation_table.tsv")
print(f"updated {n_set} fields; curated variants: {(cur_tab.final_class != '').sum()} / {len(cur_tab)}")

## 7. Genotype-positive cohort

Uses `final_class` where curation is done. Uncurated variants count as *provisional* positive if they are T1 or T2. Only QC-passing genotypes count.

- `strict`: curated P/LP (falls back to provisional T1/T2 until curation is finished)
- `broad`: strict plus T3 and uncurated "check" variants (sensitivity analysis)

In [ ]:
cur = load("curation_table.tsv")
cur["final_class"] = cur.final_class.fillna("").str.upper().str.strip()
cur["curated"] = cur.final_class != ""
cur["strict"] = np.where(cur.curated, cur.final_class.isin(["P", "LP"]), cur.tier.isin(["T1", "T2"]))
cur["broad"]  = cur.strict | (~cur.curated & cur.tier.isin(["T3", "check"])) | cur.final_class.eq("VUS")

c = carriers[carriers.qc_pass].merge(cur[["vkey", "gene", "tier", "final_class", "curated", "strict", "broad"]], on="vkey")
person = (c.groupby("person_id")
           .agg(genes=("gene", lambda s: ";".join(sorted(set(s)))),
                variants=("vkey", lambda s: ";".join(sorted(set(s)))),
                strict=("strict", "any"), broad=("broad", "any"),
                all_curated=("curated", "all"))
           .reset_index())
person = person[person.broad]

try:   # flag people in the related-sample removal list (keep them; decide per analysis)
    rel = set(read_gs_table(RELATED_PATH, dtype=str).iloc[:, 0])
    person["related_flagged"] = person.person_id.isin(rel)
except Exception as e:
    print("relatedness file not read:", e)
try:
    anc = read_gs_table(ANCESTRY_PATH, dtype=str)
    person = person.merge(anc.iloc[:, :2].set_axis(["person_id", "ancestry_pred"], axis=1), on="person_id", how="left")
except Exception as e:
    print("ancestry file not read:", e)

save(person, "hht_genotype_positive.tsv")
print("strict:", mask(int(person.strict.sum())), "| broad:", mask(int(person.broad.sum())),
      "| all variants curated:", bool(person.all_curated.all()))

In [ ]:
# Shareable summary (counts 1-20 masked)
tab = person.assign(gene=person.genes.str.split(";")).explode("gene")
out = tab.groupby("gene").agg(strict=("strict", "sum"), broad=("broad", "sum")).astype(int)
print(out.apply(lambda col: col.map(mask)))

## 8. Using the cohort in the R notebooks

```r
system(paste0("gsutil -u ", Sys.getenv("GOOGLE_PROJECT"), " cp ",
              Sys.getenv("WORKSPACE_BUCKET"), "/hht_variant_curation/hht_genotype_positive.tsv ."))
geno <- readr::read_tsv("hht_genotype_positive.tsv", col_types = readr::cols(person_id = "c"))
```

Then join on `person_id` with the diagnosed-HHT cohort from `Get_cohort.ipynb`.

**Not covered here:** structural variants (exon-level deletions/duplications in ENG/ACVRL1). Check the srWGS SV callset for those separately and add carriers to the curation table by hand.